# Economic Trade-Agreement Ties Development

This notebook normalizes economic agreement memberships, reconstructs 2000-2003 from official effective dates, preserves Darren's original signed weights as provenance, and applies a bounded family-level score.

## Baseline formula

```text
0.60 × bilateral comprehensive FTA
+ 0.35 × any regional trade agreement
+ 0.05 × sectoral trade agreement
```

Additional agreements within the same family remain visible but do not increase the baseline without limit.

## Pre-2004 reconstruction

The source workbook begins in 2004. The preceding four years are reconstructed from official dates:

- US-Israel FTA: active throughout 2000-2003;
- NAFTA with Canada and Mexico: active throughout 2000-2003;
- US-Jordan FTA: active at the 2001 year-end and thereafter;
- APTA with China and the five existing participating states: active from China's 2001 accession.

Every reconstructed row is marked in the normalized register.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

def locate_repository_root():
    try:
        start = Path.cwd().resolve()
    except FileNotFoundError:
        start = Path.home()
    candidates = [start]
    candidates.extend(start.parents)
    for candidate in candidates:
        if (candidate / "src" / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

REPOSITORY_ROOT = locate_repository_root()
print("Repository root located.")

def run_script(script):
    result = subprocess.run([sys.executable, str(script)], cwd=REPOSITORY_ROOT, text=True, capture_output=True)
    if result.stdout.strip(): print(result.stdout)
    if result.stderr.strip(): print("SCRIPT STDERR:"); print(result.stderr)
    if result.returncode != 0: raise RuntimeError(f"{script.name} failed with exit code {result.returncode}")

Repository root located.


In [2]:
SCRIPT = REPOSITORY_ROOT / "src" / "data" / "build_trade_agreement_ties.py"
OUTPUT = REPOSITORY_ROOT / "data" / "processed" / "trade_agreement_ties_country_anchor_year.csv"
REPORTS = REPOSITORY_ROOT / "outputs" / "tables" / "trade_agreement_ties"
run_script(SCRIPT)

ECONOMIC TRADE-AGREEMENT PRODUCTION PANEL
{
  "rows": 9612,
  "countries_or_historical_entities": 199,
  "study_years": [
    2000,
    2024
  ],
  "normalized_membership_records": 868,
  "officially_reconstructed_records": 30,
  "score_weights": {
    "bilateral_comprehensive_fta": 0.6,
    "regional_trade_agreement": 0.35,
    "sectoral_trade_agreement": 0.05
  },
  "score_distribution": {
    "0.0": 8828,
    "0.35": 399,
    "0.6": 349,
    "0.65": 2,
    "0.95": 34
  },
  "validation_failures": {
    "duplicate_keys": 0,
    "blank_country_names": 0,
    "unresolved_register_countries": 0,
    "original_weight_mismatches": 0,
    "score_formula_mismatches": 0,
    "score_bound_failures": 0,
    "source_unavailable_rows": 0
  }
}

Production output: data/processed/trade_agreement_ties_country_anchor_year.csv
Reports directory: outputs/tables/trade_agreement_ties



In [3]:
panel=pd.read_csv(OUTPUT,low_memory=False)
register=pd.read_csv(REPORTS/"trade_agreement_membership_normalized.csv",low_memory=False)
summary=pd.read_json(REPORTS/"trade_agreement_panel_summary.json",typ="series")
print("Rows:",len(panel)); print("Membership records:",len(register)); display(summary.to_frame("value"))

Rows: 9612
Membership records: 868


,value
rows,9612
countries_or_historical_entities,199
study_years,"[2000, 2024]"
normalized_membership_records,868
officially_reconstructed_records,30
score_weights,{'bilateral_comprehensive_fta': 0.600000000000...
score_distribution,"{'0.0': 8828, '0.35': 399, '0.6': 349, '0.65':..."
validation_failures,"{'duplicate_keys': 0, 'blank_country_names': 0..."


In [4]:
keys=["country_iso3","anchor","year"]
assert len(panel)==9612
assert not panel.duplicated(keys).any()
assert panel["country_name"].notna().all()
assert panel["trade_agreement_source_available"].all()
assert panel["trade_agreement_score_raw"].between(0,1).all()
expected=0.60*panel["bilateral_fta_active"].astype(int)+0.35*panel["regional_fta_active"].astype(int)+0.05*panel["sectoral_trade_agreement_active"].astype(int)
assert (panel["trade_agreement_score_raw"]-expected).abs().lt(1e-12).all()
print("PASS: complete 2000-2024 coverage")
print("PASS: score formula and bounds")

PASS: complete 2000-2024 coverage
PASS: score formula and bounds


In [5]:
reconstructed=register[register["record_source"].eq("official_date_reconstruction")]
display(reconstructed.sort_values(["year","anchor","country_iso3","agreement_id"]))
print("Reconstructed membership rows:",len(reconstructed))

,country_source_name,country_iso3,anchor,year,agreement_name,agreement_id,agreement_family,member,original_signed_weight,original_weight_magnitude,expected_original_weight,record_source,source_note
94,CAN,CAN,United States,2000,usmca_nafta,usmca_nafta,regional_trade_agreement,True,1.0,1.0,1,official_date_reconstruction,USTR NAFTA; entered into force 1994
334,ISR,ISR,United States,2000,us_bilateral_fta,us_bilateral_fta,bilateral_comprehensive_fta,True,2.0,2.0,2,official_date_reconstruction,USTR Israel FTA; entered into force 1985
546,MEX,MEX,United States,2000,usmca_nafta,usmca_nafta,regional_trade_agreement,True,1.0,1.0,1,official_date_reconstruction,USTR NAFTA; entered into force 1994
33,BGD,BGD,China,2001,apta,apta,regional_trade_agreement,True,-1.0,1.0,-1,official_date_reconstruction,UN ESCAP APTA; China acceded in 2001
299,IND,IND,China,2001,apta,apta,regional_trade_agreement,True,-1.0,1.0,-1,official_date_reconstruction,UN ESCAP APTA; China acceded in 2001
414,KOR,KOR,China,2001,apta,apta,regional_trade_agreement,True,-1.0,1.0,-1,official_date_reconstruction,UN ESCAP APTA; China acceded in 2001
464,LAO,LAO,China,2001,apta,apta,regional_trade_agreement,True,-1.0,1.0,-1,official_date_reconstruction,UN ESCAP APTA; China acceded in 2001
506,LKA,LKA,China,2001,apta,apta,regional_trade_agreement,True,-1.0,1.0,-1,official_date_reconstruction,UN ESCAP APTA; China acceded in 2001
95,CAN,CAN,United States,2001,usmca_nafta,usmca_nafta,regional_trade_agreement,True,1.0,1.0,1,official_date_reconstruction,USTR NAFTA; entered into force 1994
335,ISR,ISR,United States,2001,us_bilateral_fta,us_bilateral_fta,bilateral_comprehensive_fta,True,2.0,2.0,2,official_date_reconstruction,USTR Israel FTA; entered into force 1985


Reconstructed membership rows: 30


In [6]:
latest=panel[panel["year"].eq(2024)]
display(latest.groupby("anchor").agg(countries=("country_iso3","size"),any_agreement=("any_trade_agreement_active","sum"),bilateral=("bilateral_fta_active","sum"),regional=("regional_fta_active","sum"),sectoral=("sectoral_trade_agreement_active","sum"),mean_score=("trade_agreement_score_raw","mean")).reset_index())

,anchor,countries,any_agreement,bilateral,regional,sectoral,mean_score
0,China,196,28,15,18,0,0.078061
1,United States,196,21,13,8,1,0.054337


In [7]:
frequency=panel["trade_agreement_score_raw"].value_counts().sort_index().rename_axis("score").reset_index(name="country_anchor_years")
overlap=panel[panel["active_trade_agreement_count"].gt(1)]
display(frequency)
print("Multiple-agreement country-anchor-years:",len(overlap))
display(overlap[["country_name","anchor","year","active_trade_agreement_ids","trade_agreement_score_raw"]].tail(40))

,score,country_anchor_years
0,0.00,8828
1,0.35,399
2,0.60,349
3,0.65,2
4,0.95,34


Multiple-agreement country-anchor-years: 72


,country_name,anchor,year,active_trade_agreement_ids,trade_agreement_score_raw
4725,Laos,China,2018,apta|china_asean_fta,0.35
4726,Laos,China,2019,apta|china_asean_fta,0.35
4727,Laos,China,2020,apta|china_asean_fta,0.35
4728,Laos,China,2021,apta|china_asean_fta,0.35
4729,Laos,China,2022,apta|china_asean_fta|rcep,0.35
4730,Laos,China,2023,apta|china_asean_fta|rcep,0.35
4731,Laos,China,2024,apta|china_asean_fta|rcep,0.35
5749,Myanmar,China,2022,china_asean_fta|rcep,0.35
5750,Myanmar,China,2023,china_asean_fta|rcep,0.35
5751,Myanmar,China,2024,china_asean_fta|rcep,0.35


## Conclusion

The trade-agreement score is now production-complete for 2000-2024. It preserves bilateral, regional, and sectoral evidence separately, records overlap, and retains the original signed weights for robustness.